# App-Ready Model Training

This notebook trains simpler pipelines for product use, reducing the number of inputs that a final Streamlit user would need to provide.

## Targets

1. `expected_delay_days`
2. `adjusted_eta_days`
3. `freight_cost_index`
4. `delay_class`

All models share the same input schema and are saved as fitted pipelines in `models/app_ready/`.

## Input Design

The training schema keeps user-facing inputs compact. Instead of using direct delay components such as `weather_delay_days` and `port_handling_days`, the notebook converts them into simplified scenario levels that match the kind of controls a user could set in an app.

### Shared Inputs

- `departure_timestamp`
- `origin_port`
- `destination_port`
- `origin_region`
- `destination_region`
- `route_corridor`
- `ocean_side`
- `vessel_type`
- `cargo_type`
- `distance_nm`
- `baseline_eta_days`
- `weather_risk_level`
- `port_congestion_level`
- `geopolitical_risk_level`

In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
from joblib import dump
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
DATA_PATH = Path("../data/processed/synthetic_shipments.csv")
MODELS_DIR = Path("../models/app_ready")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
df = pd.read_csv(DATA_PATH)

def to_level(series, bins):
    return pd.cut(series, bins=bins, labels=[0, 1, 2, 3], include_lowest=True).astype(int)

df["weather_risk_level"] = to_level(df["weather_delay_days"], [-0.01, 0.15, 0.70, 1.20, np.inf])
df["port_congestion_level"] = to_level(df["port_handling_days"], [0.39, 0.75, 1.10, 1.45, np.inf])
df["geopolitical_risk_level"] = to_level(df["geopolitical_delay_days"], [-0.01, 0.0, 1.50, 3.00, np.inf])

print(f"Dataset shape: {df.shape}")
df[["weather_risk_level", "port_congestion_level", "geopolitical_risk_level"]].describe().T

Dataset shape: (12000, 27)


,count,mean,std,min,25%,50%,75%,max
weather_risk_level,12000.0,1.218083,0.991601,0.0,0.0,1.0,2.0,3.0
port_congestion_level,12000.0,1.488833,1.113094,0.0,0.0,1.0,2.0,3.0
geopolitical_risk_level,12000.0,0.369083,0.768490,0.0,0.0,0.0,0.0,3.0


## Shared Feature Set

Every model uses the same fields so the future app can submit one consistent payload regardless of the selected target.

In [3]:
BASE_FEATURES = [
    "departure_timestamp",
    "origin_port",
    "destination_port",
    "origin_region",
    "destination_region",
    "route_corridor",
    "ocean_side",
    "vessel_type",
    "cargo_type",
    "distance_nm",
    "baseline_eta_days",
    "weather_risk_level",
    "port_congestion_level",
    "geopolitical_risk_level",
]

REGRESSION_TARGETS = [
    "expected_delay_days",
    "adjusted_eta_days",
    "freight_cost_index",
]
CLASSIFICATION_TARGET = "delay_class"

NUMERIC_FEATURES = [
    "distance_nm",
    "baseline_eta_days",
    "weather_risk_level",
    "port_congestion_level",
    "geopolitical_risk_level",
    "departure_month",
    "departure_dayofweek",
    "departure_hour",
]

CATEGORICAL_FEATURES = [
    "origin_port",
    "destination_port",
    "origin_region",
    "destination_region",
    "route_corridor",
    "ocean_side",
    "vessel_type",
    "cargo_type",
]

def add_time_features(frame):
    enriched = frame.copy()
    departure_ts = pd.to_datetime(enriched["departure_timestamp"], errors="coerce")
    enriched["departure_month"] = departure_ts.dt.month.fillna(0).astype(int)
    enriched["departure_dayofweek"] = departure_ts.dt.dayofweek.fillna(0).astype(int)
    enriched["departure_hour"] = departure_ts.dt.hour.fillna(0).astype(int)
    return enriched.drop(columns=["departure_timestamp"])

X = df[BASE_FEATURES].copy()
train_idx, test_idx = train_test_split(
    df.index,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=df[CLASSIFICATION_TARGET],
)

X_train = X.loc[train_idx]
X_test = X.loc[test_idx]

print("Shared input columns:")
print(BASE_FEATURES)

Shared input columns:
['departure_timestamp', 'origin_port', 'destination_port', 'origin_region', 'destination_region', 'route_corridor', 'ocean_side', 'vessel_type', 'cargo_type', 'distance_nm', 'baseline_eta_days', 'weather_risk_level', 'port_congestion_level', 'geopolitical_risk_level']


In [4]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, NUMERIC_FEATURES),
    ("categorical", categorical_pipeline, CATEGORICAL_FEATURES),
])

def build_pipeline(model):
    return Pipeline(steps=[
        ("time_features", FunctionTransformer(add_time_features, validate=False)),
        ("preprocessor", preprocessor),
        ("model", model),
    ])

## Regression Training

Three Ridge regressors are trained using the same shared schema.

In [5]:
regression_results = []
saved_model_paths = {}

for target in REGRESSION_TARGETS:
    y_train = df.loc[train_idx, target]
    y_test = df.loc[test_idx, target]

    pipeline = build_pipeline(Ridge(alpha=1.0))
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)

    metrics = {
        "target": target,
        "mae": round(mean_absolute_error(y_test, predictions), 4),
        "rmse": round(float(np.sqrt(mean_squared_error(y_test, predictions))), 4),
        "r2": round(r2_score(y_test, predictions), 4),
    }
    regression_results.append(metrics)

    model_path = MODELS_DIR / f"{target}_pipeline.joblib"
    dump(pipeline, model_path)
    saved_model_paths[target] = str(model_path.resolve())

pd.DataFrame(regression_results)

,target,mae,rmse,r2
0,expected_delay_days,0.1801,0.2412,0.9481
1,adjusted_eta_days,0.1802,0.2412,0.9989
2,freight_cost_index,0.0064,0.0112,0.9565


## Classification Training

A logistic regression pipeline is trained for `delay_class` using the same app-ready schema.

In [6]:
y_train_cls = df.loc[train_idx, CLASSIFICATION_TARGET]
y_test_cls = df.loc[test_idx, CLASSIFICATION_TARGET]

classification_pipeline = build_pipeline(
    LogisticRegression(max_iter=2000, solver="saga", random_state=RANDOM_STATE)
)
classification_pipeline.fit(X_train, y_train_cls)
class_predictions = classification_pipeline.predict(X_test)

classification_results = pd.DataFrame([
    {
        "target": CLASSIFICATION_TARGET,
        "accuracy": round(accuracy_score(y_test_cls, class_predictions), 4),
        "macro_f1": round(f1_score(y_test_cls, class_predictions, average="macro"), 4),
    }
])

classification_model_path = MODELS_DIR / "delay_class_pipeline.joblib"
dump(classification_pipeline, classification_model_path)
saved_model_paths[CLASSIFICATION_TARGET] = str(classification_model_path.resolve())

classification_results

,target,accuracy,macro_f1
0,delay_class,0.9025,0.8666


## Save Metadata

Feature definitions, metrics, and model paths are saved for downstream app integration.

In [7]:
metadata = {
    "random_state": RANDOM_STATE,
    "base_features": BASE_FEATURES,
    "numeric_features_after_time_expansion": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "regression_targets": REGRESSION_TARGETS,
    "classification_target": CLASSIFICATION_TARGET,
    "regression_metrics": regression_results,
    "classification_metrics": classification_results.to_dict(orient="records"),
    "risk_level_mapping": {
        "weather_risk_level": "0=low/none, 1=light, 2=moderate, 3=high",
        "port_congestion_level": "0=low, 1=light, 2=moderate, 3=high",
        "geopolitical_risk_level": "0=none, 1=low, 2=medium, 3=high"
    },
    "saved_model_paths": saved_model_paths,
}

metadata_path = MODELS_DIR / "training_metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print("Saved artifacts:")
for name, path in saved_model_paths.items():
    print(f"- {name}: {path}")
print(f"- metadata: {metadata_path.resolve()}")

display(pd.DataFrame(regression_results))
display(classification_results)

Saved artifacts:
- expected_delay_days: C:\Users\filip\GitHub\Chain-Pulse-AI\models\app_ready\expected_delay_days_pipeline.joblib
- adjusted_eta_days: C:\Users\filip\GitHub\Chain-Pulse-AI\models\app_ready\adjusted_eta_days_pipeline.joblib
- freight_cost_index: C:\Users\filip\GitHub\Chain-Pulse-AI\models\app_ready\freight_cost_index_pipeline.joblib
- delay_class: C:\Users\filip\GitHub\Chain-Pulse-AI\models\app_ready\delay_class_pipeline.joblib
- metadata: C:\Users\filip\GitHub\Chain-Pulse-AI\models\app_ready\training_metadata.json


,target,mae,rmse,r2
0,expected_delay_days,0.1801,0.2412,0.9481
1,adjusted_eta_days,0.1802,0.2412,0.9989
2,freight_cost_index,0.0064,0.0112,0.9565


,target,accuracy,macro_f1
0,delay_class,0.9025,0.8666
